IMPORTS

In [18]:
from gensim.models import Word2Vec
import numpy as np
from nltk import word_tokenize
import pandas as pd

LOADING DATA

In [19]:
df = pd.read_csv("TechReviews.csv")

TOKENIZATION

In [20]:
tokens = [
    [
        w
        for w in word_tokenize(sentence.lower()) + ["<UNK>"]
        if w.isalpha() or w == "<UNK>"
    ]
    for sentence in df['Review']
]


MODEL

In [21]:
cbow = Word2Vec(tokens,window=5,vector_size=100,min_count=1,sg=0)
skipgram = Word2Vec(tokens,window=5,vector_size=100,min_count=1,sg=1)

WORD PAIR

In [22]:
word_pairs = [("battery","life")]

COSINE SIMILARITY

In [23]:
def cosine_similarity(w1,w2,model):

    if w1 not in model.wv:
        w1 = "<UNK>"
    
    if w2 not in model.wv:
        w2 = "<UNK>"

    v1 = model.wv[w1] 
    v2 = model.wv[w2] 

    return np.dot(v1,v2) / (np.linalg.norm(v1)*np.linalg.norm(v2))

    

COMPUTING COSINE SIMILARITY

In [24]:
for w1,w2 in word_pairs:
    sim1 = cosine_similarity(w1,w2,cbow)
    sim2 = cosine_similarity(w1,w2,skipgram)

    print(f"CBOW similarity between '{w1}' and '{w2}': {sim1:.4f}")
    print(f"Skip-gram similarity between '{w1}' and '{w2}': {sim2:.4f}")

CBOW similarity between 'battery' and 'life': -0.0697
Skip-gram similarity between 'battery' and 'life': -0.0655


LISTING MOST SIMILAR WORDS

In [25]:
print(cbow.wv.most_similar("camera",topn=5))

[('interface', 0.24532486498355865), ('design', 0.2305486798286438), ('stylus', 0.22091466188430786), ('pen', 0.21248698234558105), ('customer', 0.18162551522254944)]


In [26]:
print(skipgram.wv.most_similar("camera",topn=5))

[('interface', 0.24550223350524902), ('design', 0.22927819192409515), ('stylus', 0.223506361246109), ('pen', 0.21399295330047607), ('a', 0.18493176996707916)]


SAVING MODEL

In [27]:
cbow.save("cbow.model")
skipgram.save("skipgram.model")

LOADING MODEL

In [28]:
loaded_cbow = Word2Vec.load("cbow.model")
loaded_skipgram = Word2Vec.load("skipgram.model")